In [1]:
import requests
from urllib.parse import urljoin
import pandas as pd
from selectolax.parser import HTMLParser
import re
from pprint import pprint
from tqdm import tqdm
import json
from time import sleep
import io
import numpy as np

In [2]:
# Set base target platform domain for reference reconstruction
domain = "https://www.supercarworld.com"

In [3]:
def get_page (url, return_response=False):
    """
    Fetches a web page and parses its HTML content using Selectolax HTMLParser.
    
    Parameters:
    -----------
    url : str
        The URL path of the webpage to fetch.
    return_response : bool, default False
        Determines whether to return the raw HTTP response object along with the parsed page.
        
    Returns:
    --------
    HTMLParser or tuple
        Parsed HTML structural object, or a tuple consisting of (HTMLParser, requests.Response).
    """
    response = requests.get(url= url)
    web_page = HTMLParser(html= response.text)
    if return_response:
        return web_page, response
    else:
        return web_page

In [4]:
def car_crawler(car_link):
    """
    Crawls a specific car target URL to parse its HTML and extract its trailing numeric identification sequence.
    
    Parameters:
    -----------
    car_link : str
        The destination profile URL string.
        
    Returns:
    --------
    tuple
        A tuple structured as (car_id [str], web_page [HTMLParser]).
    """
    car_id = re.search(r'(\d+)$', car_link)[0]
    response = requests.get(car_link)
    web_page = HTMLParser(html= response.text)
    return  car_id, web_page

def get_name_prod_year(web_page):
    """
    Extracts the car model title string and production text metrics using defined CSS paths.
    
    Parameters:
    -----------
    web_page : HTMLParser
        The parsed HTML state context.
        
    Returns:
    --------
    tuple
        A pair containing (car_name [str], production_year_range [str]).
    """
    name_css = "body > table > tbody > tr:nth-child(2) > td:nth-child(2) > table > tbody > tr:nth-child(1) > td > table > tbody > tr > td.heading"
    prod_year_css = "body > table > tbody > tr:nth-child(2) > td:nth-child(2) > table > tbody > tr:nth-child(1) > td > table > tbody > tr > td.heading > span"
    name = web_page.css(name_css)[0].text()
    prod_year = web_page.css(prod_year_css)[0].text()
    return name, prod_year
    
def get_car_details(car_id, df):
    """
    Parses unstructured layout spec metrics from index 7 of the tables DataFrame collection.
    Distributes data cleanly across General, Engine, Performance, Dimensions, and Pricing categories.
    
    Parameters:
    -----------
    car_id : str
        The specific record reference tracker key.
    df : list of pandas.DataFrame
        Collection of dataframes fetched out of the page layouts.
        
    Returns:
    --------
    tuple
        Five separate category metadata dictionaries matching structural technical fields.
    """
    # Dividing the Car info into the following segments: General, Engine, Performance, Dimensions, Pricing 
    raw_data = df[7]
    general, engine, performance, dimensions, pricing = {}, {}, {}, {}, {}
    car_info = {
        "General": general,
        "Engine": engine,
        "Performance": performance,
        "Dimensions": dimensions,
        "Pricing": pricing
    }
    for k,v in raw_data.set_index(0).T.to_dict("records")[0].items():
        if k == v:
            df_name = k
            car_info[df_name]["Car ID"]= car_id
        else:
            car_info[df_name][k] = v
    return general, engine, performance, dimensions, pricing        
    
 
def get_car_ratting(car_id, df):
    """
    Pulls score attributes, structural review traits, and rank positions from layout table index 9.
    
    Parameters:
    -----------
    car_id : str
        The operational ID lookup code key.
    df : list of pandas.DataFrame
        Collection of dataframes parsed out of the layout elements.
        
    Returns:
    --------
    tuple
        Two maps: (individual_criteria_ratings [dict], summarized_overall_metrics [dict]).
    """
    # Performance, Handling, Looks, Features, Quality, Costs, Desire, Eco, Practicality, Sound
    try:
        raw_ratings1 = df[9].loc[1:10,[0,1]]
        rating = raw_ratings1.set_index(0).T.to_dict("records")[0]
    except:
        rating = {
            'Costs': np.nan,
            'Desire': np.nan,
            'Eco': np.nan,
            'Features': np.nan,
            'Handling': np.nan,
            'Looks': np.nan,
            'Performance': np.nan,
            'Practicality': np.nan,
            'Quality': np.nan,
            'Sound': np.nan
            }
    # Overall Rating, Money No Object, Bargain Rating
    try:
        raw_ratings2 = df[9].loc[11:,0]
        cleaned_raw_rating2 = [i for i in raw_ratings2.to_list() if "Ranks" not in str(i)]
        ratings2_k = []
        ratings2_v = []
        g = 0
        for i in cleaned_raw_rating2:
            if g == 0:
                ratings2_k.append(i)
                g = 1
            else:
                g = 0
                ratings2_v.append(i)
        overall_rating = dict(zip(ratings2_k, ratings2_v))
    except:
        overall_rating = {
            'Overall Rating': np.nan,
            'Money No Object': np.nan,
            'Bargain Rating': np.nan
        }
    rating["Car ID"] = car_id
    overall_rating["Car ID"] = car_id
    return rating, overall_rating   

def get_basic_info(car_id, name, prod_year, pricing, domain):
    """
    Consolidates core baseline metrics and computes structural static image file pathways.
    
    Parameters:
    -----------
    car_id : str
        Target individual identity marker code.
    name : str
        The vehicle profile title identifier string.
    prod_year : str
        Extracted production frame timelines data.
    pricing : dict
        Market values map containing new and used evaluations metrics.
    domain : str
        The parent platform layout endpoint string.
        
    Returns:
    --------
    dict
        A dictionary holding a simplified, standardized vehicle indexing structure row.
    """
    return {
        "Car ID": car_id,
        "Name": name,
        "Production": prod_year,
        "New Price": pricing["New Price"],
        "Used Price": pricing["Used Price"],
        "Image URL": f"{domain}/cgi-bin/showcarpic.cgi?carid={car_id}&picnum=1"
    }

def extract_car_info(car_id, web_page, domain):
    """
    Orchestrates execution of tracking, structural reading, and spec partitioning across data pipelines.
    
    Parameters:
    -----------
    car_id : str
        Target tracking identity reference key.
    web_page : HTMLParser
        Parsed markup layer dataset of the current item profile context.
    domain : str
        Base destination site address root path string.
        
    Returns:
    --------
    tuple
        Deconstructed target maps mapping out individual structural dimensions parameters data blocks.
    """
    name, prod_year = get_name_prod_year(web_page)
    # Wrap the HTML string in a StringIO object
    html_file = io.StringIO(web_page.html)
    # Read the HTML into a DataFrame
    df = pd.read_html(html_file)
    general, engine, performance, dimensions, pricing = get_car_details(car_id, df)
    basic_info = get_basic_info(car_id, name, prod_year, pricing, domain)
    rating, overall_rating = get_car_ratting(car_id, df)
    
    return basic_info, rating , overall_rating, general, engine, performance, dimensions

In [5]:
def get_brands(web_page, domain):
    """
    Parses automaker brands, initial evaluation records, pathways parameters, and logo layout locations.
    
    Parameters:
    -----------
    web_page : HTMLParser
        The collection index main catalog markup layout state tracker context.
    domain : str
        Target parent web framework domain address routing path.
        
    Returns:
    --------
    dict
        A dict structure holding columns layout arrays mapping manufacturer properties.
    """
    brand_and_first_car = {}
    brand_and_first_car["Brand ID"] = []
    brand_and_first_car["Brand"] = []
    brand_and_first_car["First Car"] = []
    brand_and_first_car["Logo"] = []
    brand_id = 0
    for brand_e in web_page.css('tr > td[height="50"] > a[name]'):
        brand_id +=1
        brand_and_first_car["Brand ID"].append(brand_id)
        brand_and_first_car["Brand"].append(brand_e.attributes.get("name"))
        brand_and_first_car["First Car"].append(domain+"/cgi-bin/"+brand_e.parent.parent.next.css('td > a')[0].attributes.get("href"))
        brand_and_first_car["Logo"].append(domain+"/images/logos/place_hoder.gif".replace("place_hoder",brand_e.attributes.get("name").replace(" ","%20")))
    return brand_and_first_car
    
def get_car_links(web_page, response):
    """
    Compiles absolute pathways points pointing toward underlying profile sub-nodes within catalog grids.
    
    Parameters:
    -----------
    web_page : HTMLParser
        The targeted structural webpage representation parsed layer.
    response : requests.Response
        The core tracking execution network session metadata layout containing origin path state traits.
        
    Returns:
    --------
    list
        Array listing clean absolute destination URL string variables pointing to each individual car.
    """
    return [urljoin(response.url, e.attributes['href']) for e in web_page.css('[href*="showgeneral.cgi?"]')]
    # return [domain + e.attributes['href'] for e in web_page.css('[href*="showgeneral.cgi?"])]

In [6]:
# Execute initial connection to load the entire inventory sequence catalog layout
web_page, response = get_page(domain+"/cgi-bin/listall.cgi?make", True)

In [7]:
# Build manufacturer profile rows array layout structures from parsed layout values map tracking definitions
brands = get_brands(web_page, domain)

In [8]:
countries = []
flags = []
country_element = ' table > tbody > tr > td:nth-child(3) > a > img'

# Iterate through manufacturers tracking anchors to parse origin country and geographic asset parameters details
for car in tqdm(brands["First Car"]):
    web_page = get_page(car)
    flag = domain + web_page.css(country_element)[0].attributes.get("src")
    country = flag.split("/")[-1].split(".")[0].replace("%20"," ")
    countries.append(country)
    flags.append(flag)
    sleep(1.5)  # to be polite to the server
    
# Substitute the processing initialization links placeholder array track columns with target tracking arrays
del brands["First Car"]
brands["Flag"] = flags
brands["Country"] = countries

100%|██████████| 244/244 [08:42<00:00,  2.14s/it]


In [9]:
# Save structured automaker profile catalog directly down onto localized disk CSV format sheet
pd.DataFrame(brands).to_csv("brands.csv", index = False)

In [10]:
# Gather full array path list capturing every distinctive individual automobile webpage locator element line item
cars_links = get_car_links(web_page, response)
len(cars_links)

1196

In [12]:
# Apply standard cool-down rest delay spacing to mitigate rapid continuous requests hitting network limits
sleep(300)

In [ ]:
# Instantiating temporary record storage containers to isolate structured metadata segments
basic_info_list = []
rating_list = []    
overall_rating_list = []    
general_list = []
engine_list = []    
performance_list = []   
dimensions_list = []    

# Deploy main deep harvesting iterative scraping iteration block handling retry routines and connection parameters
for link in tqdm(cars_links):
    trials = 0
    while True:
        try:
            # Extract deep internal metadata assets blocks tracking structure mapping arrays properties
            car_id, web_page = car_crawler(link)
            basic_info, rating , overall_rating, general, engine, performance, dimensions = extract_car_info(car_id, web_page, domain)
            
            # Appending matching structural profiles components records safely into storage repositories
            basic_info_list.append(basic_info)
            rating_list.append(rating)
            overall_rating_list.append(overall_rating)
            general_list.append(general)
            engine_list.append(engine)
            performance_list.append(performance)
            dimensions_list.append(dimensions)
            sleep(1.5)  # to be polite to the server
            break
        except Exception as e:
            trials += 1
            if trials > 1:
                raise(e)
            else:
                sleep(62)
                break

 70%|██████▉   | 833/1196 [27:28<12:01,  1.99s/it]C:\Users\Abdalrhman\AppData\Local\Temp\ipykernel_8504\3619860635.py:38: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rating = raw_ratings1.set_index(0).T.to_dict("records")[0]
100%|██████████| 1196/1196 [39:32<00:00,  1.98s/it]


In [14]:
# Export compiled distinct staging entity lists cleanly out into structured individual tabular datasets
pd.DataFrame(basic_info_list).to_csv("cars_basic_info.csv", index=False)
pd.DataFrame(rating_list).to_csv("cars_rating.csv", index=False)
pd.DataFrame(overall_rating_list).to_csv("cars_overall_rating.csv", index=False)
pd.DataFrame(general_list).to_csv("cars_general.csv", index=False)
pd.DataFrame(engine_list).to_csv("cars_engine.csv", index=False)
pd.DataFrame(performance_list).to_csv("cars_performance.csv", index=False)
pd.DataFrame(dimensions_list).to_csv("cars_dimensions.csv", index=False)